# Notebook 03 — RAG End-to-End (the LangChain way)

**Workshop:** Agentic AI for Scientists — Week 2
**Block:** 4 of 6 (35 min)
**Goal:** RAG (Retrieval-Augmented Generation) is two halves: **retrieval** and **prompt augmentation**. We build the whole pipeline with LangChain's native pieces — loaders, splitters, embeddings, vector stores — then compare three retrieval strategies on five real questions.

**Corpus:** five public ML/AI papers (downloaded on first run).

**Pipeline — and the LangChain class for each stage:**

```
PDFs --PyPDFLoader-->  Documents
      --RecursiveCharacterTextSplitter-->  chunks
      --HuggingFaceEmbeddings-->  vectors
      --FAISS / Chroma-->  vector store --.as_retriever()--> retriever
                                                                 |
question --retriever--> top-k chunks --prompt--> Gemini --> grounded answer
```

We retrieve three ways — **dense** (FAISS), **sparse** (BM25), **hybrid** (EnsembleRetriever) — then score `hit@5` on a labelled eval set.


## 0. Setup

In [ ]:
%pip install -q \
    "google-genai>=1.0" \
    "langchain==0.3.7" "langchain-google-genai==2.0.11" "langchain-community==0.3.5" \
    "langchain-text-splitters==0.3.2" "langchain-huggingface==0.1.2" "langchain-chroma==0.1.4" \
    "faiss-cpu==1.9.0" "rank-bm25==0.2.2" "sentence-transformers==3.3.0" \
    "pypdf==5.1.0" "pandas==2.2.3" python-dotenv requests


In [ ]:
import os

# Free Gemini API key (no credit card): https://aistudio.google.com/apikey
# Colab: add GOOGLE_API_KEY under the key icon (left sidebar) -> "Secrets".
# Local: put GOOGLE_API_KEY=AIza... in a .env file next to this notebook.
try:
    from google.colab import userdata  # type: ignore
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()

assert os.environ.get("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY first (see the comment above)."
print("API key loaded.")


In [ ]:
import os
os.environ["ANONYMIZED_TELEMETRY"] = "False"   # silence Chroma's telemetry chatter
from pathlib import Path
import json, requests
import pandas as pd

MODEL = "gemini-2.5-flash"
print("Ready.")


---

## 1. Download the corpus

Five public ML/AI papers from arXiv. ~7 MB total.


In [ ]:
CORPUS_DIR = Path("sample_articles")
CORPUS_DIR.mkdir(exist_ok=True)

PAPERS = {
    "react_yao_2022.pdf": "https://arxiv.org/pdf/2210.03629",
    "attention_vaswani_2017.pdf": "https://arxiv.org/pdf/1706.03762",
    "chinchilla_hoffmann_2022.pdf": "https://arxiv.org/pdf/2203.15556",
    "constitutional_ai_bai_2022.pdf": "https://arxiv.org/pdf/2212.08073",
    "rag_lewis_2020.pdf": "https://arxiv.org/pdf/2005.11401",
}

for filename, url in PAPERS.items():
    target = CORPUS_DIR / filename
    if target.exists() and target.stat().st_size > 10_000:
        print(f"  cached: {filename}")
        continue
    print(f"  downloading: {filename}")
    r = requests.get(url, timeout=60, headers={"User-Agent": "agentic-workshop/1.0"})
    r.raise_for_status()
    target.write_bytes(r.content)
print("\nDone.")


---

## 2. Load PDFs with `PyPDFLoader`

LangChain's loaders all return a list of **`Document`** objects — `.page_content` (the text) plus `.metadata` (here: `source` file and `page` number). One Document per page. This metadata is what lets us cite a source later.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pages = []
for path in sorted(CORPUS_DIR.glob("*.pdf")):
    loaded = PyPDFLoader(str(path)).load()      # list[Document], one per page
    # Normalise 'source' to just the filename so citations stay short.
    for d in loaded:
        d.metadata["source"] = path.name
    pages.extend(loaded)
    print(f"  {path.name}: {len(loaded)} pages")

print(f"\nTotal: {len(pages)} page-documents")
print("\n--- sample Document ---")
print("metadata:", pages[0].metadata)
print("content :", pages[0].page_content[:200].replace(chr(10), ' '), "...")


---

## 3. Chunk with `RecursiveCharacterTextSplitter`

A page is too big to embed as one vector. We split into ~800-character chunks. The **recursive** splitter tries paragraph -> sentence -> word -> char boundaries in order, so chunks rarely cut mid-sentence. Use `.split_documents()` (not `.split_text()`) so the per-page metadata is carried onto every chunk.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,          # ~200 tokens; fits comfortably when we stuff 5 chunks into context
    chunk_overlap=100,       # ~12% overlap so a sentence split across chunks survives in one of them
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(pages)

# Tag each chunk with a short id we can cite, e.g. react_yao_2022.pdf#42
for i, c in enumerate(chunks):
    c.metadata["chunk_id"] = f"{c.metadata['source']}#{i}"

print(f"{len(pages)} pages -> {len(chunks)} chunks")
print("\n--- sample chunk ---")
print("chunk_id:", chunks[0].metadata["chunk_id"])
print(chunks[0].page_content[:250], "...")


> **The dirty secret of RAG:** chunk size/overlap often matters more than which embedding model you pick. We use 800/100 as a sane default; the closing exercises sweep it. **Semantic chunking** (splitting on embedding-distance jumps) exists via `SemanticChunker` — slower, marginal gains here, so we skip it live.

---

## 4. Embeddings — a free local model

`all-MiniLM-L6-v2` is 90 MB, runs on CPU in Colab's free tier, and is plenty for a 5-paper demo. `HuggingFaceEmbeddings` (from `langchain_huggingface`) wraps it in LangChain's embedding interface so any vector store can use it. Production swap: `text-embedding-3-small` (OpenAI) or `text-embedding-004` (Google).


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Sanity check: embed one string, look at the dimension
v = embeddings.embed_query("What is the ReAct loop?")
print(f"embedding dimension: {len(v)}")


---

## 5. Dense retrieval — FAISS vector store

`FAISS.from_documents` embeds every chunk and builds an in-memory similarity index in one call. `.as_retriever()` hands back a standard LangChain **retriever** — a component whose only job is `query -> list[Document]`, which plugs straight into a chain.


In [ ]:
from langchain_community.vectorstores import FAISS

faiss_store = FAISS.from_documents(chunks, embeddings)
faiss_retriever = faiss_store.as_retriever(search_kwargs={"k": 5})
print(f"FAISS index built: {faiss_store.index.ntotal} vectors")

# Look at what comes back
hits = faiss_retriever.invoke("What is the ReAct loop?")
for d in hits[:3]:
    print(f"  {d.metadata['chunk_id']}: {d.page_content[:110].replace(chr(10),' ')}...")


You can persist a FAISS index to disk and reload it (handy so you don't re-embed every run):

```python
faiss_store.save_local("faiss_index")
reloaded = FAISS.load_local("faiss_index", embeddings,
                            allow_dangerous_deserialization=True)  # required: the index is unpickled
```

The `allow_dangerous_deserialization=True` flag is mandatory on load — unpickling can execute code, so only load indexes you built yourself.

---

## 6. The same store, persisted — Chroma

FAISS is a raw in-memory index. **Chroma** is a small local *database*: pass `persist_directory=` and it writes to disk and **auto-persists** (no `.persist()` call needed in modern `langchain_chroma`). Reopen the directory later and your vectors are still there — useful when the corpus is expensive to embed.


In [ ]:
from langchain_chroma import Chroma

chroma_store = Chroma.from_documents(
    chunks, embeddings,
    persist_directory="./chroma_db",
    collection_name="w2_papers",
)
chroma_retriever = chroma_store.as_retriever(search_kwargs={"k": 5})
print(f"Chroma collection size: {chroma_store._collection.count()} vectors")

# Same retriever interface, different backend — identical call shape
hits = chroma_retriever.invoke("What is the ReAct loop?")
for d in hits[:3]:
    print(f"  {d.metadata['chunk_id']}: {d.page_content[:110].replace(chr(10),' ')}...")


**FAISS vs Chroma, in one line:** FAISS = fastest pure-vector index, lives in RAM; Chroma = a persistent local store with metadata filtering and a friendlier API. Both expose the *same* `.as_retriever()` interface, so the rest of the notebook doesn't care which you picked. We use the FAISS retriever for the comparison below.

---

## 7. Sparse retrieval — BM25

BM25 is classical keyword scoring — no embeddings, fast, and surprisingly strong on exact-term queries (acronyms, names, jargon). `BM25Retriever.from_documents` builds it from the same chunks, giving us the same `query -> Documents` interface.


In [ ]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 5

hits = bm25_retriever.invoke("What is the ReAct loop?")
for d in hits[:3]:
    print(f"  {d.metadata['chunk_id']}: {d.page_content[:110].replace(chr(10),' ')}...")


---

## 8. Hybrid retrieval — `EnsembleRetriever`

Dense catches *paraphrase* ("what does X mean?"); sparse catches *exact terms* ("RLAIF", "DPR"). `EnsembleRetriever` fuses both with **Reciprocal Rank Fusion** (RRF) — it merges by rank, not raw score, so we sidestep the normalization headache entirely. `weights` bias the fusion.


In [ ]:
from langchain.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.4, 0.6],     # lean slightly toward dense; tune on your eval set
)

hits = hybrid_retriever.invoke("What is the ReAct loop?")
for d in hits[:3]:
    print(f"  {d.metadata['chunk_id']}: {d.page_content[:110].replace(chr(10),' ')}...")


---

## 9. Compare the three retrievers — `hit@5` on a labelled eval set

`hit@5` = 1 if any of the top-5 chunks comes from the expected paper. The stricter **section hit** = 1 if any ground-truth keyword appears in the retrieved text (did we get the *right part* of the right paper?).


In [ ]:
# The five labelled test questions, inlined so this notebook has zero external deps.
# Each names the paper (expected_source) and a few ground-truth keywords from the
# section that answers it (expected_section_keywords).
questions = [
    {"id": "q1_react_loop",
     "question": "What are the three steps in a single iteration of the ReAct loop?",
     "expected_source": "react_yao_2022.pdf",
     "expected_section_keywords": ["Thought", "Action", "Observation", "interleaving"]},
    {"id": "q2_attention_complexity",
     "question": "Why is scaled dot-product attention preferred over additive attention for transformer architectures?",
     "expected_source": "attention_vaswani_2017.pdf",
     "expected_section_keywords": ["scaled dot-product", "additive", "matrix multiplication", "faster"]},
    {"id": "q3_chinchilla_scaling",
     "question": "According to the Chinchilla paper, what is the optimal ratio of training tokens to model parameters?",
     "expected_source": "chinchilla_hoffmann_2022.pdf",
     "expected_section_keywords": ["tokens per parameter", "compute-optimal", "scaling", "training tokens"]},
    {"id": "q4_constitutional_ai_rlaif",
     "question": "What is the role of the AI feedback signal in Constitutional AI training?",
     "expected_source": "constitutional_ai_bai_2022.pdf",
     "expected_section_keywords": ["RLAIF", "AI feedback", "preference model", "constitution", "self-critique"]},
    {"id": "q5_rag_architecture",
     "question": "What are the two main components of the retrieval-augmented generation architecture?",
     "expected_source": "rag_lewis_2020.pdf",
     "expected_section_keywords": ["retriever", "generator", "non-parametric", "parametric", "BART"]},
]
print(f"Loaded {len(questions)} test questions")


def source_hit(docs, expected_source):
    return int(any(d.metadata["source"] == expected_source for d in docs))


def section_hit(docs, keywords):
    blob = " ".join(d.page_content.lower() for d in docs)
    return int(any(kw.lower() in blob for kw in keywords))


retrievers = {"dense": faiss_retriever, "bm25": bm25_retriever, "hybrid": hybrid_retriever}

rows = []
for q in questions:
    row = {"id": q["id"], "expected": q["expected_source"].split("_")[0]}
    for name, r in retrievers.items():
        docs = r.invoke(q["question"])
        row[f"{name}_hit@5"] = source_hit(docs, q["expected_source"])
        row[f"{name}_section"] = section_hit(docs, q["expected_section_keywords"])
    rows.append(row)

df = pd.DataFrame(rows)
print(df.to_string(index=False))

print("\n=== Aggregate ===")
for col in df.columns:
    if col.endswith("hit@5") or col.endswith("section"):
        print(f"  {col:16s}: {df[col].mean():.2f}")


**Read the table.** On a corpus this small, all three usually nail the *paper-level* hit. The *section-level* hit is the honest metric. Hybrid tends to win it because it gets both the paraphrased and the exact-term queries. For your own corpus, tune the ensemble `weights` on an eval set like this one — production typically lands between 0.3 and 0.7 dense.

---

## 10. Wire a retriever into a RAG chain (the "augmentation" half)

Retrieval done — now *use* it. The LCEL pipe stuffs the retrieved chunks into the prompt's `{context}`, the model answers from them, and we ask it to **cite the chunk_id** behind every claim.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0)

RAG_PROMPT = ChatPromptTemplate.from_template("""You are a research assistant. Answer the question using ONLY the context below. If the context doesn't contain the answer, say so — do not invent. Cite the [chunk_id] in brackets after every claim.

Context:
{context}

Question: {question}

Answer (with citations):""")


def format_docs(docs):
    return "\n\n".join(f"[{d.metadata['chunk_id']}] {d.page_content}" for d in docs)


rag_chain = (
    {"context": hybrid_retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT | llm | StrOutputParser()
)

q = questions[0]["question"]
print("Q:", q, "\n")
print(rag_chain.invoke(q))


The answer should carry `[react_yao_2022.pdf#N]` citations — every claim points back to a chunk you can open and verify. That grounding is the whole reason RAG beats plain chat for research work.

> **The official shortcut.** LangChain also ships `create_retrieval_chain` + `create_stuff_documents_chain`, which do the retrieve-then-stuff wiring for you and return a dict with both the `answer` and the retrieved `context`. The explicit LCEL pipe above is the transparent version; the helper is the batteries-included one. Same idea either way.

---

## 11. RAG vs no RAG

Same question, straight to the model with no retrieval:


In [ ]:
bare = llm.invoke(questions[0]["question"]).content
print(bare)


Notice the difference: ungrounded, the model answers from training memory — no citations, and it can be confidently wrong on specifics. With RAG, every claim is anchored to a retrievable chunk.

---

## 12. Closing exercises (for after class)

1. **Chunk-size sweep.** Re-run cells 3 & 5 with `chunk_size=400` and `1600`. How does `hit@5` move?
2. **Weight sweep.** Plot `hybrid_section` as the ensemble `weights` go from `[1, 0]` (BM25 only) to `[0, 1]` (dense only). Where's the sweet spot for this corpus?
3. **Swap the store.** Replace `faiss_retriever` with `chroma_retriever` in the ensemble. Same numbers? (They should be — same vectors, same interface.)
4. **Add your own paper.** Drop a PDF into `sample_articles/`, re-run cells 2–5, ask about it. Does the answer cite the right chunk?
5. **Production embeddings.** Swap `HuggingFaceEmbeddings` for OpenAI `text-embedding-3-small`. Better section-hit?

---

**Optional next:** [Notebook 04 — Elasticsearch Appendix](04_rag_elasticsearch_appendix.ipynb)
**Week 3 preview:** this `hybrid_retriever` becomes a *tool node* in a LangGraph multi-agent researcher. Same retriever, different orchestrator.
